# Azure Infrastructure Operationalisation

> **Notebook flow:** 01 Setup → 02 EDA → 03 Train → 04 Docker → 05 Kubernetes → 06 Cleanup · **[07 Azure Deploy]** · 08 API Tests

This notebook provisions all Azure infrastructure required to run the MLOps project in production.
It follows the steps documented in [docs/operationalisation.md](../docs/operationalisation.md).

**What this notebook does:**
1. Authenticates to Azure and sets the active subscription
2. Creates a **service principal** with scoped permissions for CI/CD automation
3. Provisions all Azure resources (Resource Group → ACR → AKS → Blob Storage → App Insights)
4. Configures Kubernetes namespaces and secrets
5. Sets up ACR image retention policy
6. Validates the full deployment end-to-end

**Prerequisites:**
- Dev Container is running (Azure CLI pre-installed)
- You have an Azure subscription with **Owner** or **Contributor + User Access Administrator** role
- `az login` has been run at least once (the first cell will verify)

---

## How to use this notebook

1. **Edit the configuration cell below** with your Azure details
2. Run cells **top to bottom** — each cell depends on the previous
3. Some cells produce output you'll need for Azure DevOps setup (noted in markdown)


## 0. Configuration

Edit the values below to match your Azure environment. All subsequent cells reference these variables.

| Variable | Description |
|---|---|
| `DRY_RUN` | Set `True` to run the notebook locally without Azure — returns realistic fake responses |
| `SUBSCRIPTION_ID` | Your Azure subscription ID (from `az account list`) |
| `LOCATION` | Azure region for all resources |
| `RESOURCE_GROUP` | Resource group name |
| `ACR_NAME` | Container Registry name (must be globally unique, alphanumeric only) |
| `AKS_NAME` | Kubernetes cluster name |
| `STORAGE_ACCOUNT` | Storage account name (must be globally unique, lowercase alphanumeric) |
| `APP_INSIGHTS_NAME` | Application Insights component name |
| `SP_NAME` | Service principal display name for CI/CD |
| `API_KEY` | API key for the prediction endpoint (set a strong, random value) |
| `TRAIN_IMAGE_NAME` | Docker image name for the training container |
| `INFER_IMAGE_NAME` | Docker image name for the inference API container |

> **Tip**: Set `DRY_RUN = True` to test the full notebook flow locally without Azure credentials.
> When `DRY_RUN` is enabled, all `az` and `kubectl` commands return simulated responses so you can
> verify the variable wiring and output formatting before provisioning real resources.
>
> The values you set here must match the ADO variable group `bank-marketing-vars`.
> The summary cell at the end prints all values needed for the variable group.

In [ ]:
import os
import sys

ROOT = "/workspaces/marketing-model-mlops-azure"
os.chdir(ROOT)
sys.path.insert(0, ROOT)

# ── Deployment mode ──────────────────────────────────────────────────────
# Set True to test the notebook flow locally without any Azure resources.
# All az/kubectl commands return realistic fake responses.

DRY_RUN = True

# ── Edit these values ────────────────────────────────────────────────────

SUBSCRIPTION_ID = "<your-subscription-id>"  # az account list -o table
LOCATION = "southafricanorth"  # Azure region
RESOURCE_GROUP = "rg-bank-marketing"  # Resource group
ACR_NAME = "bankmarketingacr"  # ACR (globally unique)
AKS_NAME = "bank-marketing-aks"  # AKS cluster
AKS_NODE_COUNT = 2  # Node count
AKS_NODE_SIZE = "Standard_B2s"  # Node VM size
STORAGE_ACCOUNT = "bankmarketingdata"  # Storage (globally unique)
APP_INSIGHTS_NAME = "bank-marketing-insights"  # App Insights
SP_NAME = "sp-bank-marketing-cicd"  # Service principal
API_KEY = "<your-secure-api-key>"  # API key for /predict
TRAIN_IMAGE_NAME = "bank-marketing-train"  # Training image repo name
INFER_IMAGE_NAME = "bank-marketing-api"  # Inference image repo name

# ── Derived values (do not edit) ─────────────────────────────────────────

BLOB_CONTAINER_TRAINING = "training-data"
BLOB_CONTAINER_REGISTRY = "model-registry"
NS_PROD = "bank-marketing"
NS_DEV = "bank-marketing-dev"

# ── Activate dry-run mode in infra module ────────────────────────────────

import src.infra

src.infra.DRY_RUN = DRY_RUN

# Validate user edited the placeholders (skip in dry-run mode)
if not DRY_RUN:
    assert (
        SUBSCRIPTION_ID != "<your-subscription-id>"
    ), "⚠️  Edit SUBSCRIPTION_ID above before running. Use: az account list -o table"
    assert (
        API_KEY != "<your-secure-api-key>"
    ), "⚠️  Edit API_KEY above — set a strong random value (e.g. python -c 'import secrets; print(secrets.token_urlsafe(32))')"

if DRY_RUN:
    print("🔶 DRY-RUN MODE — no Azure resources will be created")
    print("   All az/kubectl commands return simulated responses.\n")

print("Configuration loaded:")
print(f"  Mode:              {'DRY RUN' if DRY_RUN else 'LIVE'}")
print(f"  Subscription:      {SUBSCRIPTION_ID}")
print(f"  Location:          {LOCATION}")
print(f"  Resource Group:    {RESOURCE_GROUP}")
print(f"  ACR:               {ACR_NAME}")
print(f"  AKS:               {AKS_NAME} ({AKS_NODE_COUNT}x {AKS_NODE_SIZE})")
print(f"  Storage:           {STORAGE_ACCOUNT}")
print(f"  App Insights:      {APP_INSIGHTS_NAME}")
print(f"  Service Principal: {SP_NAME}")
print(f"  Train Image:       {TRAIN_IMAGE_NAME}")
print(f"  Infer Image:       {INFER_IMAGE_NAME}")

## 1. Azure Login & Subscription

Verify you are logged in and set the correct subscription. If `az login` hasn't been run, the cell will prompt you.

In [ ]:
from src.infra import login_check, set_subscription

try:
    account = login_check()
    print(f"Logged in as: {account.get('user', {}).get('name', 'unknown')}")
    print(f"Current subscription: {account.get('name')} ({account.get('id')})")
except Exception:
    if DRY_RUN:
        raise  # Should never happen in dry-run — bug in infra.py
    print("Not logged in. Run 'az login' in the terminal first, then re-run this cell.")
    raise SystemExit(1)

# Set the target subscription
set_subscription(SUBSCRIPTION_ID)
print(f"\n✓ Active subscription set to: {SUBSCRIPTION_ID}")

## 2. Service Principal for CI/CD

A **service principal (SP)** is an identity used by Azure DevOps pipelines to deploy resources without human credentials.

### Permission requirements

The SP needs the following permissions:

| Permission | Scope | Why |
|---|---|---|
| **Contributor** | Resource Group | Create/manage ACR, AKS, Storage, App Insights |
| **AcrPush** | ACR | Push Docker images from CI pipeline |
| **Storage Blob Data Reader** | `training-data` container | Download training data in pipelines |
| **Storage Blob Data Contributor** | `model-registry` container | Upload/promote model artifacts |

### Your account requirements

To create a service principal, **your account** needs:
- **Owner** role on the subscription/resource group, OR
- **Contributor** + **User Access Administrator** (to assign RBAC roles to the SP)

> **Security note**: The SP password is shown once and cannot be retrieved later. Copy it immediately and store it in a secure location (e.g. Azure Key Vault or ADO variable group as a secret).

We create the resource group first (the SP needs a scope to be assigned to), then create the SP scoped to it.

In [ ]:
import json
from src.infra import create_resource_group, create_service_principal

# Step 1: Create the resource group (SP needs this as scope)
print("Creating resource group...")
rg_result = create_resource_group(RESOURCE_GROUP, LOCATION)
rg_info = json.loads(rg_result.stdout)
rg_id = rg_info["id"]
print(f"✓ Resource group: {RESOURCE_GROUP} (id: {rg_id})")

# Step 2: Create the service principal scoped to the resource group
print(f"\nCreating service principal '{SP_NAME}' with Contributor role...")
sp_result = create_service_principal(SP_NAME, role="Contributor", scope=rg_id)
sp_info = json.loads(sp_result.stdout)

SP_APP_ID = sp_info["appId"]
SP_PASSWORD = sp_info["password"]
SP_TENANT = sp_info["tenant"]

print(f"\n✓ Service principal created successfully")
print(f"  App ID (client ID): {SP_APP_ID}")
print(f"  Tenant ID:          {SP_TENANT}")
print(f"  Password:           {SP_PASSWORD}")
print("\n⚠️  SAVE THE PASSWORD NOW — it cannot be retrieved later.")
print("   Store it in your ADO variable group as a secret variable.")

## 3. Azure Container Registry (ACR)

The ACR stores Docker images for both the training and inference containers. The Basic SKU (10 GiB) is sufficient for this project.

In [ ]:
import json
from src.infra import create_acr, assign_role

print(f"Creating ACR: {ACR_NAME}...")
acr_result = create_acr(RESOURCE_GROUP, ACR_NAME)
acr_info = json.loads(acr_result.stdout)
acr_id = acr_info["id"]
print(f"✓ ACR created: {acr_info['loginServer']}")

# Grant the service principal AcrPush so CI can push images
print(f"\nAssigning AcrPush role to {SP_NAME}...")
assign_role(SP_APP_ID, "AcrPush", acr_id)
print(f"✓ AcrPush role assigned to SP on {ACR_NAME}")

## 4. Azure Kubernetes Service (AKS)

Creates a 2-node AKS cluster attached to the ACR. `--attach-acr` grants the AKS managed identity the `AcrPull` role — pods can pull images without `imagePullSecrets`.

> **Note**: AKS creation takes 3–5 minutes.

In [ ]:
import json
from src.infra import create_aks, get_aks_credentials, assign_role

print(f"Creating AKS cluster: {AKS_NAME} ({AKS_NODE_COUNT}x {AKS_NODE_SIZE})...")
print("This takes 3-5 minutes...")
aks_result = create_aks(
    RESOURCE_GROUP,
    AKS_NAME,
    node_count=AKS_NODE_COUNT,
    node_vm_size=AKS_NODE_SIZE,
    acr_name=ACR_NAME,
)
aks_info = json.loads(aks_result.stdout)
aks_id = aks_info["id"]
print(f"✓ AKS cluster created: {aks_info['name']}")
print(f"  Kubernetes version: {aks_info.get('kubernetesVersion', 'N/A')}")
print(f"  Node resource group: {aks_info.get('nodeResourceGroup', 'N/A')}")

# Grant the service principal AKS Cluster User so CI/CD can deploy via kubectl
print(f"\nAssigning AKS Cluster User role to {SP_NAME}...")
assign_role(SP_APP_ID, "Azure Kubernetes Service Cluster User Role", aks_id)
print(f"✓ AKS Cluster User role assigned to SP on {AKS_NAME}")

# Download credentials
print("\nDownloading kubeconfig...")
get_aks_credentials(RESOURCE_GROUP, AKS_NAME)
print("✓ kubeconfig updated — kubectl now targets the AKS cluster")

## 5. Kubernetes Namespaces

Create the two namespaces used by the project:
- `bank-marketing` — production (2 replicas, LoadBalancer)
- `bank-marketing-dev` — staging (1 replica, ClusterIP)

In [ ]:
from src.infra import create_k8s_namespace

for ns in [NS_PROD, NS_DEV]:
    result = create_k8s_namespace(ns)
    if result.returncode == 0:
        print(f"✓ Namespace created: {ns}")
    elif "AlreadyExists" in result.stderr:
        print(f"✓ Namespace already exists: {ns}")
    else:
        print(f"✗ Failed to create namespace {ns}: {result.stderr}")

## 6. Azure Blob Storage

Creates the storage account and two containers:
- `training-data` — holds versioned training CSVs
- `model-registry` — holds model artifacts with prefix-based promotion (`builds/<buildId>/` → `staging/artifacts/` → `production/artifacts/`)

Also uploads the initial training data and grants the service principal appropriate Blob RBAC roles.

In [ ]:
import json
from datetime import date
from src.infra import (
    create_storage_account,
    create_blob_container,
    upload_blob,
    copy_blob,
    assign_role,
)

# Create storage account
print(f"Creating storage account: {STORAGE_ACCOUNT}...")
sa_result = create_storage_account(STORAGE_ACCOUNT, RESOURCE_GROUP, LOCATION)
sa_info = json.loads(sa_result.stdout)
sa_id = sa_info["id"]
print(f"✓ Storage account created: {STORAGE_ACCOUNT}")

# Create containers
for container in [BLOB_CONTAINER_TRAINING, BLOB_CONTAINER_REGISTRY]:
    create_blob_container(STORAGE_ACCOUNT, container)
    print(f"✓ Blob container created: {container}")

# Upload initial training data
data_file = "data/raw/bank_marketing_data.csv"
if os.path.exists(data_file) or DRY_RUN:
    date_prefix = date.today().isoformat()
    versioned_name = f"{date_prefix}/bank_marketing_data.csv"
    print(f"\nUploading training data as {versioned_name}...")
    upload_blob(STORAGE_ACCOUNT, BLOB_CONTAINER_TRAINING, versioned_name, data_file)
    print(f"✓ Uploaded to {BLOB_CONTAINER_TRAINING}/{versioned_name}")

    # Copy to latest/
    copy_blob(
        STORAGE_ACCOUNT,
        BLOB_CONTAINER_TRAINING,
        versioned_name,
        BLOB_CONTAINER_TRAINING,
        "latest/bank_marketing_data.csv",
    )
    print(f"✓ Copied to {BLOB_CONTAINER_TRAINING}/latest/bank_marketing_data.csv")
else:
    print(f"⚠️  {data_file} not found — skipping upload (add the CSV and re-run)")

# Grant SP Blob RBAC roles
print("\nAssigning Blob storage roles to service principal...")
training_scope = f"{sa_id}/blobServices/default/containers/{BLOB_CONTAINER_TRAINING}"
registry_scope = f"{sa_id}/blobServices/default/containers/{BLOB_CONTAINER_REGISTRY}"

assign_role(SP_APP_ID, "Storage Blob Data Reader", training_scope)
print(f"✓ Storage Blob Data Reader on {BLOB_CONTAINER_TRAINING}")

assign_role(SP_APP_ID, "Storage Blob Data Contributor", registry_scope)
print(f"✓ Storage Blob Data Contributor on {BLOB_CONTAINER_REGISTRY}")

## 7. Azure Monitor & Application Insights

Enables the AKS monitoring addon and creates an Application Insights component for request tracing and prediction logging.

> **Optional for initial deployment** — you can skip this cell and add monitoring later.

In [ ]:
import json
from src.infra import enable_aks_monitoring, create_app_insights

print("Enabling AKS monitoring addon...")
enable_aks_monitoring(RESOURCE_GROUP, AKS_NAME)
print("✓ AKS monitoring enabled")

print(f"\nCreating Application Insights: {APP_INSIGHTS_NAME}...")
ai_result = create_app_insights(APP_INSIGHTS_NAME, RESOURCE_GROUP, LOCATION)
ai_info = json.loads(ai_result.stdout)
INSTRUMENTATION_KEY = ai_info.get("instrumentationKey", "N/A")
CONNECTION_STRING = ai_info.get("connectionString", "N/A")

print(f"✓ Application Insights created")
print(f"  Instrumentation Key: {INSTRUMENTATION_KEY}")
print(f"  Connection String:   {CONNECTION_STRING[:60]}...")
print(
    "\n  Save the connection string for the APPLICATIONINSIGHTS_CONNECTION_STRING env var."
)

## 8. Kubernetes Secrets

Creates secrets in both namespaces. The K8s deployment manifests reference these via `secretKeyRef`:
- `bank-marketing-api-key` — API key for `/predict` endpoint authentication
- `azure-storage` — storage account name and container for Blob model loading

In [ ]:
from src.infra import create_k8s_secret

for ns in [NS_PROD, NS_DEV]:
    print(f"\n--- Namespace: {ns} ---")

    # API key secret
    r1 = create_k8s_secret("bank-marketing-api-key", ns, {"API_KEY": API_KEY})
    if r1.returncode == 0:
        print(f"✓ Secret created: bank-marketing-api-key")
    elif "AlreadyExists" in r1.stderr:
        print(f"✓ Secret already exists: bank-marketing-api-key")
    else:
        print(f"✗ Failed: {r1.stderr.strip()}")

    # Azure storage secret
    r2 = create_k8s_secret(
        "azure-storage",
        ns,
        {
            "ACCOUNT_NAME": STORAGE_ACCOUNT,
            "CONTAINER_NAME": BLOB_CONTAINER_REGISTRY,
        },
    )
    if r2.returncode == 0:
        print(f"✓ Secret created: azure-storage")
    elif "AlreadyExists" in r2.stderr:
        print(f"✓ Secret already exists: azure-storage")
    else:
        print(f"✗ Failed: {r2.stderr.strip()}")

## 9. ACR Image Retention

Prevents the ACR Basic SKU (10 GiB) from filling up. Creates a scheduled purge task that removes build-ID-tagged training images older than 30 days while keeping `train-latest` and `latest` indefinitely.

In [ ]:
from src.infra import create_acr_purge_task

print("Creating ACR purge task...")
create_acr_purge_task(ACR_NAME)
print("✓ ACR purge task created: runs daily at 01:00 UTC")
print("  Purges: bank-marketing-train images tagged train-[0-9]+ older than 30 days")
print("  Keeps:  train-latest, latest (no numeric suffix)")

## 10. Summary & Next Steps

All Azure infrastructure is now provisioned. The cell below prints:
1. A summary of all deployed resources
2. **Every variable** needed in the ADO `bank-marketing-vars` variable group
3. The two ADO service connections you need to create

> **Important**: The variable group values printed below must match **exactly** what the three pipeline files (`.azure/azure-pipelines.yml`, `.azure/pr-validation.yml`, `.azure/retrain.yml`) expect at runtime.

In [ ]:
if DRY_RUN:
    print("=" * 66)
    print("  DRY-RUN COMPLETE — no Azure resources were created")
    print("=" * 66)
    print()
    print("  The notebook flow executed successfully with simulated responses.")
    print("  To provision real infrastructure, set DRY_RUN = False in cell 3")
    print("  and fill in your Azure details.")
    print()

print("=" * 66)
print("  INFRASTRUCTURE PROVISIONING COMPLETE")
print("=" * 66)
print()
print("Azure Resources:")
print(f"  Resource Group:     {RESOURCE_GROUP}")
print(f"  ACR:                {ACR_NAME}.azurecr.io")
print(f"  AKS:                {AKS_NAME} ({AKS_NODE_COUNT}x {AKS_NODE_SIZE})")
print(f"  Storage Account:    {STORAGE_ACCOUNT}")
print(f"  App Insights:       {APP_INSIGHTS_NAME}")
print(f"  K8s Namespaces:     {NS_PROD}, {NS_DEV}")
print()
print("Service Principal (for ADO service connection):")
print(f"  App ID:             {SP_APP_ID}")
print(f"  Tenant ID:          {SP_TENANT}")
print(f"  Password:           (saved from cell 7 above)")
print()
print("-" * 66)
print("  ADO VARIABLE GROUP: bank-marketing-vars")
print("-" * 66)
print()
print("  Copy these values into your Azure DevOps variable group.")
print("  The three pipeline YAML files reference every variable below.")
print()
print(f"  ACR_NAME                   = {ACR_NAME}")
print(f"  AKS_CLUSTER                = {AKS_NAME}")
print(f"  RESOURCE_GROUP             = {RESOURCE_GROUP}")
print(f"  BLOB_STORAGE_ACCOUNT       = {STORAGE_ACCOUNT}")
print(f"  BLOB_CONTAINER_TRAINING    = {BLOB_CONTAINER_TRAINING}")
print(f"  BLOB_CONTAINER_REGISTRY    = {BLOB_CONTAINER_REGISTRY}")
print(f"  TRAIN_IMAGE_NAME           = {TRAIN_IMAGE_NAME}")
print(f"  INFER_IMAGE_NAME           = {INFER_IMAGE_NAME}")
print()
print("  ── Service Connections (create in ADO Project Settings) ──")
print()
print("  ACR_SERVICE_CONNECTION     = <name of your ACR service connection>")
print("  AZURE_SUBSCRIPTION         = <name of your Azure RM service connection>")
print()
print("  To create these:")
print("    1. ADO → Project Settings → Service connections → New")
print("    2. 'Docker Registry' → Azure Container Registry → select your ACR")
print("       → name it and set ACR_SERVICE_CONNECTION to that name")
print("    3. 'Azure Resource Manager' → Service Principal (manual)")
print(f"       → Subscription: {SUBSCRIPTION_ID}")
print(f"       → App ID: {SP_APP_ID}, Tenant: {SP_TENANT}, Password: (from cell 7)")
print("       → name it and set AZURE_SUBSCRIPTION to that name")
print()
print("-" * 66)
print()
print("Next steps:")
print("  1. Create ADO project and service connections (see above)")
print("  2. Register pipelines (docs/operationalisation.md §10)")
print("  3. Create variable group with values above (§11)")
print("  4. Configure environments and approval gates (§12)")
print("  5. Set up GitHub branch protection (§13)")
print("  6. Run the validation sequence (§16)")